# Chapter 4: Inspect ordinary search and its continuation

**The question.** Chapter 4 compares ways of running the search with the same budget. This notebook asks the same question of the fresh run: with new seeds, did twinning starts, random starts and continuation (the twinning search given a second budget) end with different answers?

**What was run live.** Run 20260923T220909Z repeated the whole study on two machines. `spark-fe12` ran the numerical search with three new seeds (74101 to 74103), one live structural proposal from a language model and 200,000 new confirmation customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated population and the frozen money-laundering model are the same as in the `regional_case` notebooks.

**Result.** At two of the three seeds all three searches ended with the same region. Only at seed 74103 did continuation find a weaker one.

**Terms.** *Share*: the fraction of customers in a region. *Contrast*: how much worse the model does inside the region than outside it (the difference in average Brier loss); higher means a weaker spot.

The seeds, the proposal and the confirmation sample are new, so do not mix these numbers with those in the `regional_case` notebooks. This notebook reads recorded results; rerunning it makes no language-model call.

## Setup and source checks

Finds the saved results of this run and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Final answers of the ordinary searches

For each seed and search the cell counts the candidates scored and shows the final answer's `[contrast, share]` on the discovery customers.

In [2]:
rows=[]
for seed in seeds:
 for arm in ['twinning','random','continuation']:
  run=read(f'regional/{arm}_{seed}.json')
  rows.append(dict(seed=seed,arm=arm,scored=len(run['records']),objectives=run['primary']['objectives']))
display(pd.DataFrame(rows))

,seed,arm,scored,objectives
0,74101,twinning,400,"[0.06686019793661731, 0.103525]"
1,74101,random,400,"[0.06686019793661731, 0.103525]"
2,74101,continuation,800,"[0.06686019793661731, 0.103525]"
3,74102,twinning,400,"[0.06484871872744419, 0.102525]"
4,74102,random,400,"[0.06484871872744419, 0.102525]"
5,74102,continuation,800,"[0.06484871872744419, 0.102525]"
6,74103,twinning,400,"[0.062222290413305445, 0.111725]"
7,74103,random,400,"[0.06151516900522714, 0.116525]"
8,74103,continuation,800,"[0.07233707886575506, 0.1034]"


**Read the output.** **In one line: the extra search budget changed the answer at only one seed of three.**

**Columns.** `seed`: which run. `arm`: twinning start, random start or continuation. `scored`: sets of centers in the search history; continuation's 800 include the 400 it inherited from twinning. `objectives`: the final answer's contrast and share on the discovery customers.

**Notice.**

- At seeds 74101 and 74102 all three searches ended with the same region: contrast 0.066860 at 10.35% share, and contrast 0.064849 at 10.25%. The second budget found nothing better.
- At seed 74103 continuation found a narrower, weaker region (contrast 0.072337 at 10.34%) than twinning (0.062222) or random starts (0.061515).
- These are discovery numbers, which flatter each search. The Chapter 2 notebook in this folder shows the same regions measured on new customers.